# Phase 4 - Step 5: DR Grading (3 ViT Base16 9ch)


In [ ]:
# ── Installs (run once) ────────────────────────────────────────────────
!pip install timm albumentations scikit-learn --quiet

import os, json, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    cohen_kappa_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm



In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device(
    "cuda"  if torch.cuda.is_available()  else
    "mps"   if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")



In [ ]:
CFG = dict(
    img_size        = 512,
    num_channels    = 9,
    num_classes     = 5,
    seed            = 42,
    warmup_epochs   = 8,
    warmup_lr       = 3e-4,
    finetune_epochs = 50,
    lr_backbone     = 1e-5,
    lr_head         = 3e-4,
    weight_decay    = 1e-4,
    grad_clip       = 1.0,
    batch_size      = 16,
    accum_steps     = 2,
    patience        = 10,
    use_amp         = True,
    label_smoothing = 0.1,
    model_name      = "vit_base",
    results_dir     = "ViTBase_9ch_results"
)

Path(CFG["results_dir"]).mkdir(parents=True, exist_ok=True)
for sub in ["checkpoints", "plots", "predictions", "metrics"]:
    Path(f"{CFG['results_dir']}/{sub}").mkdir(exist_ok=True)
print("Results will be saved to:", CFG["results_dir"])



In [ ]:
def get_train_transforms(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.2),
        A.ColorJitter(brightness=0.15, contrast=0.15, p=0.4),
        A.GaussNoise(var_limit=(5, 20), p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.1),
        A.Normalize(
            mean=[0.485, 0.456, 0.406, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
            std= [0.229, 0.224, 0.225, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        ),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
            std= [0.229, 0.224, 0.225, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        ),
        ToTensorV2(),
    ])



In [ ]:
class DR9ChannelDataset(Dataset):
    """
    Loads a fundus image + 6 pre-generated mask PNGs directly from the clean folder 
    structure without needing a CSV file, stacking them into a [9, H, W] tensor.
    """
    def __init__(self, samples_list, root_dir, transform=None):
        self.samples = samples_list
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        filename = sample['filename']
        class_name = sample['class_name']
        label = sample['label']

        # ── Load RGB image ────────────────────────────────────────────────
        rgb_path = os.path.join(self.root_dir, 'images', class_name, filename)
        img = cv2.imread(rgb_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # (H, W, 3)

        # ── Load 6 mask PNGs as grayscale ─────────────────────────────────
        channels = [img]
        for mask_type in ["vessel", "MA", "HE", "EX", "OD", "CW"]:
            m_path = os.path.join(self.root_dir, mask_type, class_name, filename)
            m = cv2.imread(m_path, cv2.IMREAD_GRAYSCALE)  # (H, W)
            channels.append(np.expand_dims(m, axis=-1))   # → (H, W, 1)

        # ── Stack: [H,W,3] + 6×[H,W,1] = [H,W,9] ────────────────────────
        stacked = np.concatenate(channels, axis=-1).astype(np.float32)

        # ── Augment ───────────────────────────────────────────────────────
        if self.transform:
            stacked = self.transform(image=stacked)["image"]  # → (9, H, W) Tensor

        return stacked, torch.tensor(label, dtype=torch.long)



In [ ]:
# ── Define your Kaggle Input Path ─────────────────────────────────────
# (Make sure you uploaded the 'Phase_B_9Channel_Dataset.zip' as a Kaggle Dataset!)
DATA_ROOT = "/kaggle/input/phase-b-9channel-dataset" 

# ── 1. Scan folders to build the dataset in memory (NO CSV NEEDED!) ───
classes = ['0_No_DR', '1_Mild', '2_Moderate', '3_Severe', '4_Proliferative_DR']
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}

all_samples = []
base_img_dir = os.path.join(DATA_ROOT, 'images')

for cls_name in classes:
    cls_dir = os.path.join(base_img_dir, cls_name)
    if not os.path.exists(cls_dir): continue
    for filename in os.listdir(cls_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg', '.tif')):
            all_samples.append({
                'filename': filename, 
                'class_name': cls_name, 
                'label': class_to_idx[cls_name]
            })

print(f"Successfully found {len(all_samples)} total 9-channel images.")

# ── 2. Stratified Train/Val/Test split (70/15/15) ─────────────────────
from sklearn.model_selection import train_test_split

labels = [s['label'] for s in all_samples]

train_samples, temp_samples, train_labels, temp_labels = train_test_split(
    all_samples, labels, test_size=0.3, stratify=labels, random_state=42
)
val_samples, test_samples = train_test_split(
    temp_samples, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Split sizes: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

# ── 3. Compute class weights for imbalanced grades ────────────────────
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(5),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("\nClass weights:", class_weights)

# ── 4. Build datasets & loaders ───────────────────────────────────────
train_dataset = DR9ChannelDataset(train_samples, DATA_ROOT, get_train_transforms(CFG["img_size"]))
val_dataset   = DR9ChannelDataset(val_samples,   DATA_ROOT, get_val_transforms(CFG["img_size"]))
test_dataset  = DR9ChannelDataset(test_samples,  DATA_ROOT, get_val_transforms(CFG["img_size"]))

train_loader = DataLoader(
    train_dataset, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG["batch_size"]*2, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG["batch_size"]*2, shuffle=False,
    num_workers=4, pin_memory=True,
)

print(f"\nBatches per epoch: {len(train_loader)} train | {len(val_loader)} val")



In [ ]:
# ── Visual sanity check: plot one 9-channel sample ────────────────────
sample_img, sample_label = train_dataset[0]
ch_names = ["Red", "Green", "Blue", "Vessel", "MA", "Hemorrhage", "Hard Exudate", "Optic Disc", "Cotton Wool"]
colors   = ["Reds", "Greens", "Blues", "Purples", "Oranges", "RdPu", "YlOrBr", "Greens", "Greys"]

fig, axes = plt.subplots(1, 9, figsize=(27, 3))
fig.suptitle(f"9-Channel Sample | Grade: {sample_label}", fontsize=14, fontweight="bold")

for i, (ax, name, cmap) in enumerate(zip(axes, ch_names, colors)):
    ch = sample_img[i].numpy()
    ax.imshow(ch, cmap=cmap)
    ax.set_title(name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{CFG['results_dir']}/plots/channel_sanity_check.png", dpi=150)
plt.show()



In [ ]:
def build_vit_base_9ch(num_classes=5, pretrained=True):
    model = timm.create_model("vit_base_patch16_224", pretrained=pretrained,
                              num_classes=num_classes, img_size=512)
    old_proj = model.patch_embed.proj
    
    new_proj = nn.Conv2d(
        in_channels=9, out_channels=old_proj.out_channels,
        kernel_size=old_proj.kernel_size, stride=old_proj.stride,
        padding=old_proj.padding, bias=old_proj.bias is not None
    )
    
    with torch.no_grad():
        new_proj.weight[:, :3, :, :] = old_proj.weight
        new_proj.weight[:, 3:, :, :] = 0.0
        if old_proj.bias is not None:
            new_proj.bias.copy_(old_proj.bias)

    model.patch_embed.proj = new_proj
    print(f"✅ ViT-Base/16 modified: patch_embed.proj now accepts 9 channels")
    return model

model = build_vit_base_9ch(num_classes=CFG["num_classes"]).to(device)



In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CFG["label_smoothing"],
)

def compute_qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def compute_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        return 0.0



In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler, accum_steps=2):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc="Training", leave=False)
    for step, (imgs, labels) in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=CFG["use_amp"]):
            logits = model(imgs)
            loss   = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item()*accum_steps:.4f}")

    qwk = compute_qwk(all_labels, all_preds)
    return {"loss": total_loss / len(loader), "qwk": qwk}

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    for imgs, labels in tqdm(loader, desc="Evaluating", leave=False):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        probs  = F.softmax(logits, dim=1)

        total_loss  += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    return {
        "loss"      : total_loss / len(loader),
        "qwk"       : compute_qwk(y_true, y_pred),
        "auc_macro" : compute_auc(y_true, y_prob),
        "f1_macro"  : f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "preds"     : y_pred,
        "labels"    : y_true,
        "probs"     : y_prob,
    }



In [ ]:
def freeze_backbone(model, model_name):
    for name, param in model.named_parameters():
        if model_name == "efficientnet_b4":
            trainable = "conv_stem" in name or "classifier" in name
        elif model_name == "convnext_base":
            trainable = "stem.0" in name or "head" in name
        elif model_name == "vit_base":
            trainable = "patch_embed.proj" in name or "head" in name
        param.requires_grad = trainable

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Phase 1: {n_train:,} / {n_total:,} parameters are trainable ({n_train/n_total*100:.1f}%)")

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Phase 2: All {n_train:,} parameters are now trainable")

freeze_backbone(model, CFG["model_name"])

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["warmup_lr"],
    weight_decay=CFG["weight_decay"],
)
scheduler_p1 = torch.optim.lr_scheduler.ConstantLR(optimizer_p1, factor=1.0)
scaler = GradScaler(enabled=CFG["use_amp"])

history = {"phase": [], "epoch": [], "train_loss": [], "val_loss": [],
           "val_qwk": [], "val_auc": [], "val_f1_macro": [], "lr": []}
best_qwk = -1.0

print(f"\n{'='*60}")
print(f"PHASE 1: WARM-UP ({CFG['warmup_epochs']} epochs, backbone frozen)")
print(f"{'='*60}")

for epoch in range(1, CFG["warmup_epochs"] + 1):
    train_m = train_one_epoch(model, train_loader, optimizer_p1, criterion, device, scaler, CFG["accum_steps"])
    val_m   = evaluate(model, val_loader, criterion, device)
    scheduler_p1.step()

    lr = optimizer_p1.param_groups[0]["lr"]
    print(f"[P1 Epoch {epoch:02d}/{CFG['warmup_epochs']}] "
          f"Train Loss: {train_m['loss']:.4f} | Val Loss: {val_m['loss']:.4f} | "
          f"QWK: {val_m['qwk']:.4f} | AUC: {val_m['auc_macro']:.4f} | LR: {lr:.2e}")

    for k, v in [("phase","P1"),("epoch",epoch),("train_loss",train_m["loss"]),("val_loss",val_m["loss"]),
                 ("val_qwk",val_m["qwk"]),("val_auc",val_m["auc_macro"]),("val_f1_macro",val_m["f1_macro"]),("lr",lr)]:
        history[k].append(v)

    if val_m["qwk"] > best_qwk:
        best_qwk = val_m["qwk"]
        torch.save({
            "epoch": epoch, "phase": 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer_p1.state_dict(),
            "val_qwk": best_qwk, "val_auc": val_m["auc_macro"],
        }, f"{CFG['results_dir']}/checkpoints/best_warmup.pt")

print(f"\n✅ Phase 1 complete. Best Val QWK: {best_qwk:.4f}")



In [ ]:
unfreeze_all(model)

def get_param_groups(model, model_name, lr_backbone, lr_head):
    if model_name == "efficientnet_b4":
        head_params    = list(model.classifier.parameters())
        backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
    elif model_name == "convnext_base":
        head_params    = list(model.head.parameters())
        backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
    elif model_name == "vit_base":
        head_params    = list(model.head.parameters())
        backbone_params = [p for n, p in model.named_parameters() if "head" not in n]

    return [
        {"params": backbone_params, "lr": lr_backbone, "name": "backbone"},
        {"params": head_params,     "lr": lr_head,     "name": "head"},
    ]

optimizer_p2 = torch.optim.AdamW(
    get_param_groups(model, CFG["model_name"], CFG["lr_backbone"], CFG["lr_head"]),
    weight_decay=CFG["weight_decay"],
)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=CFG["finetune_epochs"], eta_min=1e-7
)

class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, mode="max"):
        self.patience, self.min_delta, self.mode = patience, min_delta, mode
        self.best = -float("inf") if mode == "max" else float("inf")
        self.counter = 0

    def step(self, metric):
        improved = (metric > self.best + self.min_delta) if self.mode == "max" else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

early_stop = EarlyStopping(patience=CFG["patience"], mode="max")

print(f"\n{'='*60}")
print(f"PHASE 2: FINE-TUNING ({CFG['finetune_epochs']} epochs, differential LR)")
print(f"{'='*60}")

for epoch in range(1, CFG["finetune_epochs"] + 1):
    train_m = train_one_epoch(model, train_loader, optimizer_p2, criterion, device, scaler, CFG["accum_steps"])
    val_m   = evaluate(model, val_loader, criterion, device)
    scheduler_p2.step()

    lr_bb = optimizer_p2.param_groups[0]["lr"]
    print(f"[P2 Epoch {epoch:02d}/{CFG['finetune_epochs']}] "
          f"Train Loss: {train_m['loss']:.4f} | Val Loss: {val_m['loss']:.4f} | "
          f"QWK: {val_m['qwk']:.4f} | AUC: {val_m['auc_macro']:.4f} | F1: {val_m['f1_macro']:.4f} | BB-LR: {lr_bb:.2e}")

    for k, v in [("phase","P2"),("epoch",epoch),("train_loss",train_m["loss"]),("val_loss",val_m["loss"]),
                 ("val_qwk",val_m["qwk"]),("val_auc",val_m["auc_macro"]),("val_f1_macro",val_m["f1_macro"]),("lr",lr_bb)]:
        history[k].append(v)

    if val_m["qwk"] > best_qwk:
        best_qwk = val_m["qwk"]
        torch.save({
            "epoch": epoch, "phase": 2,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer_p2.state_dict(),
            "val_qwk": best_qwk, "val_auc": val_m["auc_macro"],
            "val_f1_macro": val_m["f1_macro"], "cfg": CFG,
        }, f"{CFG['results_dir']}/checkpoints/best_model.pt")
        print(f"  💾 New best model saved (QWK={best_qwk:.4f})")

    if early_stop.step(val_m["qwk"]):
        print(f"\n⏹ Early stopping at epoch {epoch}. Best QWK: {best_qwk:.4f}")
        break

print(f"\n✅ Phase 2 complete. Best Val QWK: {best_qwk:.4f}")



In [ ]:
print("\nLoading best checkpoint for test evaluation...")
ckpt = torch.load(f"{CFG['results_dir']}/checkpoints/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (phase {ckpt['phase']}), QWK at save: {ckpt['val_qwk']:.4f}")

test_m = evaluate(model, test_loader, criterion, device)

print(f"\n{'='*60}\nTEST SET RESULTS — {CFG['model_name'].upper()}\n{'='*60}")
print(f"  QWK (primary)  : {test_m['qwk']:.4f}")
print(f"  AUC macro OvR  : {test_m['auc_macro']:.4f}")
print(f"  F1 macro       : {test_m['f1_macro']:.4f}\n{'='*60}")

results_json = {
    "model_name": CFG["model_name"], "test_qwk": test_m["qwk"],
    "test_auc_macro": test_m["auc_macro"], "test_f1_macro": test_m["f1_macro"],
    "best_val_qwk": ckpt["val_qwk"], "best_epoch": ckpt["epoch"],
}
with open(f"{CFG['results_dir']}/metrics/test_results.json", "w") as f:
    json.dump(results_json, f, indent=2)



In [ ]:
history_df = pd.DataFrame(history)
y_true, y_pred, y_prob = test_m["labels"], test_m["preds"], test_m["probs"]
class_names = ["Grade 0\n(None)", "Grade 1\n(Mild)", "Grade 2\n(Moderate)", "Grade 3\n(Severe)", "Grade 4\n(Prolif.)"]

fig = plt.figure(figsize=(24, 18))
fig.suptitle(f"{CFG['model_name'].upper()} — DR Grading Results Dashboard", fontsize=18, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :2])
p1_hist, p2_hist = history_df[history_df["phase"] == "P1"], history_df[history_df["phase"] == "P2"]
e_p1, e_p2 = range(1, len(p1_hist)+1), range(len(p1_hist)+1, len(p1_hist)+len(p2_hist)+1)
ax1.plot(e_p1, p1_hist["train_loss"], "b-o", ms=3, label="Train Loss (P1)")
ax1.plot(e_p1, p1_hist["val_loss"],   "b--s", ms=3, label="Val Loss (P1)")
ax1.plot(e_p2, p2_hist["train_loss"], "g-o", ms=3, label="Train Loss (P2)")
ax1.plot(e_p2, p2_hist["val_loss"],   "g--s", ms=3, label="Val Loss (P2)")
ax1.axvline(x=len(p1_hist)+0.5, color="red", linestyle=":", alpha=0.7)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

ax2 = fig.add_subplot(gs[0, 2])
all_e = list(e_p1) + list(e_p2)
all_q = list(p1_hist["val_qwk"]) + list(p2_hist["val_qwk"])
ax2.plot(all_e, all_q, "purple", marker="o", ms=3)
ax2.axhline(y=0.85, color="red", linestyle="--", alpha=0.7)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("QWK")

ax3 = fig.add_subplot(gs[1, :2])
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax3, xticklabels=class_names, yticklabels=class_names)
ax3.set_xlabel("Predicted Grade"); ax3.set_ylabel("True Grade")

ax4 = fig.add_subplot(gs[1, 2])
colors_roc = ["#E24B4A", "#EF9F27", "#639922", "#185FA5", "#7F77DD"]
for i, (name, col) in enumerate(zip(class_names, colors_roc)):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_prob[:, i])
    auc_i = roc_auc_score((y_true == i).astype(int), y_prob[:, i])
    ax4.plot(fpr, tpr, color=col, label=f"{name.split(chr(10))[0]} (AUC={auc_i:.3f})")
ax4.legend(fontsize=7)

ax5 = fig.add_subplot(gs[2, :])
metrics_names  = ["QWK", "AUC Macro", "F1 Macro", "F1 Weighted"]
metrics_vals   = [test_m["qwk"], test_m["auc_macro"], test_m["f1_macro"], test_m["f1_weighted"]]
targets        = [0.85, 0.95, 0.80, 0.80]
bar_colors     = ["#639922" if v >= t else "#E24B4A" for v, t in zip(metrics_vals, targets)]
bars = ax5.bar(metrics_names, metrics_vals, color=bar_colors, alpha=0.8, width=0.5)
for bar, val, tgt in zip(bars, metrics_vals, targets):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{val:.4f}", ha="center", va="bottom", fontweight="bold")
    ax5.axhline(y=tgt, color="red", linestyle="--")
ax5.set_ylim(0, 1.1)

plt.savefig(f"{CFG['results_dir']}/plots/results_dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

